# Actividad: las cuatro normalizaciones a mano

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe-tec/machine-learning/blob/main/activities/activity_normalization.ipynb)

En esta actividad programas Batch Norm, Layer Norm, Instance Norm y Group Norm con operaciones básicas de PyTorch y compruebas que coinciden con las capas de la biblioteca. Después mides por qué Batch Norm falla con batches pequeños.

Completa cada celda marcada con `# TODO` y ejecuta la celda de verificación que sigue: imprime **Correcto** cuando tu código funciona.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
x = torch.arange(32.).reshape(2, 4, 2, 2)   # (N, C, H, W): 2 imágenes, 4 canales, 2 × 2 posiciones
print(x)

## Parte 1. Una sola fórmula

Las cuatro técnicas aplican la misma operación:

$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

Lo único que cambia es sobre qué ejes se calculan $\mu$ y $\sigma^2$. Escribe una función `normalize(x, dims)` que normalice sobre los ejes `dims` y conserve la forma de `x`. Usa la varianza sin corrección de Bessel (`unbiased=False`), como lo hacen las capas de PyTorch.

In [ ]:
def normalize(x, dims, eps=1e-5):
    # TODO: resta la media y divide entre la raíz de la varianza más eps.
    # Pista: usa x.mean(dim=dims, keepdim=True) y x.var(dim=dims, keepdim=True, unbiased=False).
    return None

In [ ]:
# Verificación
try:
    z = normalize(x, dims=(2, 3))
    assert z is not None, "normalize todavía regresa None."
    assert z.shape == x.shape, f"La forma debe ser {tuple(x.shape)} y es {tuple(z.shape)}."
    assert torch.allclose(z.mean(dim=(2, 3)), torch.zeros(2, 4), atol=1e-5), "Cada grupo debe quedar con media 0."
    assert torch.allclose(z.std(dim=(2, 3), unbiased=False), torch.ones(2, 4), atol=1e-3), "Cada grupo debe quedar con desviación 1."
    print("Correcto: cada grupo queda con media 0 y desviación 1.")
except AssertionError as e:
    print("Revisa tu código:", e)

## Parte 2. Batch, Layer e Instance Norm

Con `normalize` ya tienes casi todo. Para un tensor de forma (N, C, H, W), escribe los ejes que promedia cada técnica:

- **Batch Norm:** una media por canal, usando todas las imágenes del batch.
- **Layer Norm:** una media por imagen, usando todos sus canales.
- **Instance Norm:** una media por imagen y por canal.

In [ ]:
# TODO: escribe los ejes que promedia cada normalización para un tensor (N, C, H, W).
DIMS_BATCH = None      # p. ej. (0, 1)
DIMS_LAYER = None
DIMS_INSTANCE = None

In [ ]:
# Verificación: compara con las funciones de PyTorch (sin γ ni β).
def check(name, dims, reference):
    if dims is None:
        print(f"{name}: todavía falta escribir los ejes.")
        return
    mine = normalize(x, dims=dims)
    if mine is not None and torch.allclose(mine, reference, atol=1e-4):
        print(f"Correcto: {name} coincide con PyTorch usando dims={dims}.")
    else:
        print(f"Revisa {name}: con dims={dims} no coincide con PyTorch.")

check("Batch Norm", DIMS_BATCH, F.batch_norm(x, None, None, training=True))
check("Layer Norm", DIMS_LAYER, F.layer_norm(x, x.shape[1:]))
check("Instance Norm", DIMS_INSTANCE, F.instance_norm(x))

## Parte 3. Group Norm

Group Norm divide los C canales en G grupos y normaliza cada grupo de una imagen por separado. El truco es cambiar la forma del tensor para que cada grupo quede en su propio eje.

In [ ]:
def group_norm(x, G, eps=1e-5):
    N, C, H, W = x.shape
    # TODO: agrupa los canales con x.reshape(N, G, C // G, H, W),
    # normaliza cada grupo con normalize(...) y regresa a la forma (N, C, H, W).
    return None

In [ ]:
# Verificación
try:
    for G in (1, 2, 4):
        mine = group_norm(x, G)
        assert mine is not None, "group_norm todavía regresa None."
        assert mine.shape == x.shape, f"La forma debe ser {tuple(x.shape)}."
        assert torch.allclose(mine, F.group_norm(x, G), atol=1e-4), f"No coincide con PyTorch para G = {G}."
    assert torch.allclose(group_norm(x, 1), normalize(x, (1, 2, 3)), atol=1e-4)
    assert torch.allclose(group_norm(x, 4), normalize(x, (2, 3)), atol=1e-4)
    print("Correcto: group_norm coincide con PyTorch. Con G = 1 es Layer Norm y con G = C es Instance Norm.")
except (AssertionError, TypeError) as e:
    print("Revisa tu código:", e)

## Parte 4. ¿Por qué falla Batch Norm con batches pequeños?

Batch Norm estima la media de cada canal con las imágenes del batch. Si el batch es pequeño, esa estimación es ruidosa. Para medirlo, generas un conjunto de 10,000 imágenes sintéticas: cada imagen tiene su propio brillo, así que las imágenes difieren entre sí. La media real del canal 0 es `true_mean`.

In [ ]:
torch.manual_seed(0)
brightness = torch.randn(10_000, 1, 1, 1) * 2.0 + 5.0      # cada imagen tiene su propio brillo
data = brightness + 0.5 * torch.randn(10_000, 4, 8, 8)      # (N, C, H, W)
true_mean = data[:, 0].mean().item()
print(f"Media real del canal 0: {true_mean:.3f}")

In [ ]:
def batch_mean_error(B, trials=2000):
    errors = []
    for _ in range(trials):
        idx = torch.randint(0, len(data), (B,))
        # TODO: calcula la media del canal 0 como lo haría Batch Norm con el batch data[idx]
        # (promedio sobre las imágenes y las posiciones) y guarda su error absoluto contra true_mean.
        mu_batch = None
        if mu_batch is None:
            return None
        errors.append(abs(mu_batch - true_mean))
    return sum(errors) / len(errors)

In [ ]:
# Verificación y experimento
torch.manual_seed(1)
try:
    sizes = [32, 16, 8, 4, 2]
    errors = [batch_mean_error(B) for B in sizes]
    assert None not in errors, "mu_batch todavía es None."
    assert errors[-1] > 3 * errors[0], "Con B = 2 el error debería ser varias veces mayor que con B = 32."
    for B, e in zip(sizes, errors):
        print(f"B = {B:2d}: error medio de la media del batch = {e:.3f}")
    plt.figure(figsize=(5, 3))
    plt.plot([str(B) for B in sizes], errors, marker="o")
    plt.xlabel("tamaño del batch")
    plt.ylabel("error de la media estimada")
    plt.tight_layout()
    plt.show()
    print("Correcto: con batches pequeños, la media que usa Batch Norm se aleja mucho de la real.")
except (TypeError, AssertionError) as e:
    print("Revisa tu código:", e)

Group Norm no tiene este problema: sus medias salen de una sola imagen. Comprueba que la salida de una imagen es la misma, venga sola o acompañada.

In [ ]:
one = data[:1]
alone = F.group_norm(one, 2)
in_batch = F.group_norm(data[:32], 2)[:1]
print("Group Norm, misma salida sola o en batch:", torch.allclose(alone, in_batch, atol=1e-5))

bn_alone = F.batch_norm(one, None, None, training=True)
bn_in_batch = F.batch_norm(data[:32], None, None, training=True)[:1]
print("Batch Norm, misma salida sola o en batch:", torch.allclose(bn_alone, bn_in_batch, atol=1e-5))

## Preguntas

1. En ImageNet, una ResNet-50 con Batch Norm pasa de 23.6 % de error de validación con 32 imágenes por GPU a 34.7 % con 2 (Wu y He, 2018, *Group Normalization*, tabla 2). ¿Cómo explica tu experimento de la Parte 4 ese resultado?
2. ¿Por qué Batch Norm necesita guardar estadísticas acumuladas para la inferencia y Layer Norm no?
3. Vas a segmentar imágenes médicas de 512 × 512 y solo te caben 2 por GPU. ¿Qué normalización usarías y por qué?

*Escribe tus respuestas aquí.*